### First set up the MCP server: python3 accuweather_mcp.py

In [1]:
import operator
import random
from typing import Annotated, Sequence, TypedDict

from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, ToolMessage

from langgraph.managed.is_last_step import RemainingSteps
from langgraph.prebuilt import create_react_agent
from langgraph.errors import GraphRecursionError

from langchain_mcp_adapters.client import MultiServerMCPClient

In [2]:
LOCAL_LLM = 'gemma4:12b-mlx'
TEMPERATURE = 0.7

llm_model = ChatOllama(
    model=LOCAL_LLM,
    temperature=TEMPERATURE,
    use_responses_api=True
)

In [3]:
EMBEDDING_MODEL = 'nomic-embed-text:latest'
SAVE_DIR = './chroma_travel_db'

vectorstore_client = Chroma(
    persist_directory=SAVE_DIR,
    embedding_function=OllamaEmbeddings(model=EMBEDDING_MODEL)
)
retriever = vectorstore_client.as_retriever()

In [4]:
@tool(description='Search travel information about destinations in England.')
def search_travel_info(query: str) -> str:
    """Search embedded WikiVoyage content for 
    information about destinations in England."""    
    docs = retriever.invoke(query)
    top = docs[:4] if isinstance(docs, list) else docs
    return "\n---\n".join(d.page_content for d in top)
search_tool = [search_travel_info]

mcp_client = MultiServerMCPClient({
    "accuweather": {
        "url": "http://127.0.0.1:8020/accu-mcp-server",
        "transport": "streamable_http"
    }
})
weather_tool = await mcp_client.get_tools()

tools = search_tool + weather_tool

In [5]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    remaining_steps: RemainingSteps

In [6]:
system_prompt = '''
You are a helpful travel assistant that searches information and retrieves weather forecasts.    

CRITICAL RULES:
    1. Only suggest destinations that you have found inside the 'search_travel_info' tool.
    2. Identify candidate towns from your travel info search and check the weather for MULTIPLE candidate towns in parallel (simultaneously) to find the ones with the best weather.
    3. If your initial batch of towns has bad weather, query the weather for any backup towns in a single batch before formulating your final answer.
'''
travel_info_agent = create_react_agent(
    model=llm_model,
    tools=tools,
    state_schema=AgentState,
    prompt=system_prompt,
)

/var/folders/qp/9vxvmncx0ks8cprx94py8fdh0000gn/T/ipykernel_21594/3786618434.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  travel_info_agent = create_react_agent(


In [7]:
async def chat_loop():
    print("UK Travel Assistant (type 'exit' to quit)")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            break
        state = {"messages": [HumanMessage(content=user_input)]}
        
        # 10 steps is more than enough for a search + weather fallback batch
        config = {"recursion_limit": 10} 
        
        try:
            result = await travel_info_agent.ainvoke(state, config=config)
            print("\n\n\n")
            print(result)
            print("\n\n\n")
            response_msg = result["messages"][-1].content
            print(f"Assistant: {response_msg}\n")
        except GraphRecursionError:
            # Catch the limit gracefully if it hits a runaway loop
            print("\nAssistant: I'm sorry, I couldn't find any towns with ideal weather after checking several options.\n")


In [8]:
await chat_loop()

UK Travel Assistant (type 'exit' to quit)


You:  Suggest two Cornwall beach towns with nice weather.






{'messages': [HumanMessage(content='Suggest two Cornwall beach towns with nice weather.', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-mlx', 'created_at': '2026-06-13T23:44:26.128428Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4530196667, 'load_duration': 31367208, 'prompt_eval_count': 230, 'prompt_eval_duration': 145283833, 'eval_count': 133, 'eval_duration': 4351972459, 'logprobs': None, 'model_name': 'gemma4:12b-mlx', 'model_provider': 'ollama'}, id='lc_run--019ec35f-465b-7da1-9fda-2f2fb5802986-0', tool_calls=[{'name': 'search_travel_info', 'args': {'query': 'beach towns in Cornwall'}, 'id': '9ef77efe-8c9b-4d35-9fcf-3616e3f85649', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 230, 'output_tokens': 133, 'total_tokens': 363}), ToolMessage(content='Cornwall.jpg|300px]]\\&quot;}}&quot;}}">3</a></span> <span id="Falmouth" class="fn org listing-name"><a rel

You:  exit
